# LPatchTST — Kaggle 2x T4 GPU Notebook

This notebook handles repository setup, data preparation, multi-GPU DDP training, and downstream evaluation on Kaggle.

### Pre-requisites:
1. **Internet** must be turned **ON** in the right-hand settings panel.
2. **Accelerator** must be set to **GPU T4 x2**.

In [19]:
%%bash
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Clone Repo & Set Up CSV Dataset                           ║
# ╚══════════════════════════════════════════════════════════════════════╝
REPO_DIR="/kaggle/working/Lpatchtst"

if [ -d "$REPO_DIR" ]; then
    echo "Repository already exists. Updating to latest main..."
    cd "$REPO_DIR"
    git fetch origin
    git reset --hard origin/main
    git submodule update --init --recursive
else
    echo "Cloning repository (including submodules)..."
    git clone --recurse-submodules https://github.com/ayan1-git/Lpatchtst "$REPO_DIR"
fi

# Install lightweight required libraries (torch/numpy/pandas are pre-installed)
pip install -q einops safetensors scikit-learn

cd "$REPO_DIR"
mkdir -p Data

# Check if the cloned repository already contains CSV files in Data/
csv_count=$(ls Data/*.csv 2>/dev/null | wc -l)

if [ "$csv_count" -gt 0 ]; then
    echo "Found $csv_count pre-packaged CSV files in Data/ directory (cloned from Git)."
    echo "Using pre-packaged dataset for training."
    ls -lh Data/
else
    # Auto-discover any Kaggle input folder containing CSV files and link them
    echo "No CSV files found in cloned Data/ folder. Searching under /kaggle/input/..."
    FOUND=0
    for d in /kaggle/input/*; do
        if [ -d "$d" ]; then
            if ls "$d"/*.csv >/dev/null 2>&1; then
                echo "-> Found CSV files in $d. Symlinking to Data/..."
                ln -sf "$d"/*.csv Data/
                FOUND=1
            fi
        fi
    done
    if [ $FOUND -eq 0 ]; then
        echo "⚠️ Warning: No CSV files found. Please copy your CSVs to $REPO_DIR/Data/."
    else
        echo "Data folder contents successfully mapped from Kaggle inputs:"
        ls -lh Data/
    fi
fi
echo "✅ Setup complete!"

Repository already exists. Updating to latest main...
HEAD is now at 55cebbd ch
Found 19 pre-packaged CSV files in Data/ directory (cloned from Git).
Using pre-packaged dataset for training.
total 28M
-rw-r--r-- 1 root root 1.9M May 27 10:56 NIFTY 100_30minute.csv
-rw-r--r-- 1 root root 1.9M May 27 10:56 NIFTY 200_30minute.csv
-rw-r--r-- 1 root root 1.2M May 27 10:56 NIFTY 500_30minute.csv
-rw-r--r-- 1 root root 1.9M May 27 10:56 NIFTY 50_30minute.csv
-rw-r--r-- 1 root root 1.2M May 27 10:56 NIFTY ALPHA 50_30minute.csv
-rw-r--r-- 1 root root 1.9M May 27 10:56 NIFTY AUTO_30minute.csv
-rw-r--r-- 1 root root 2.0M May 27 10:56 NIFTY BANK_30minute (1).csv
-rw-r--r-- 1 root root 1.7M May 27 10:56 NIFTY COMMODITIES_30minute.csv
-rw-r--r-- 1 root root 633K May 27 10:56 NIFTY CONSR DURBL_30minute.csv
-rw-r--r-- 1 root root 1.4M May 27 10:56 NIFTY CONSUMPTION_30minute.csv
-rw-r--r-- 1 root root 1.1M May 27 10:56 NIFTY CPSE_30minute.csv
-rw-r--r-- 1 root root 2.0M May 27 10:56 NIFTY ENERGY_30minu

From https://github.com/ayan1-git/Lpatchtst
   e011bc0..55cebbd  main       -> origin/main


In [21]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Run DDP Training on 2x T4 GPUs via torchrun               ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys, torch

REPO_DIR = "/kaggle/working/Lpatchtst"
os.chdir(REPO_DIR)

# 1. Force unbuffered output and resolve NCCL binding hangs in subprocesses
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["NCCL_SOCKET_IFNAME"] = "lo"

num_gpus = torch.cuda.device_count()
print(f"Detecting GPUs: {num_gpus} active")

cmd = f"torchrun --standalone --nnodes=1 --nproc_per_node={num_gpus} --master_port=29500 train.py"
print(f"Running command: {cmd}\n")
print("-" * 70)

# 2. Launch process and stream stdout+stderr line-by-line in real-time
process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    cwd=REPO_DIR
)

try:
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n⚠️ Interrupted! Terminating training process…")
    process.terminate()
    process.wait()

rc = process.poll()
if rc != 0:
    raise RuntimeError(f"Training failed with exit code {rc}")
print("\n✅ Training complete successfully!")

Detecting GPUs: 2 active
Running command: torchrun --standalone --nnodes=1 --nproc_per_node=2 --master_port=29500 train.py

----------------------------------------------------------------------
W0527 16:17:28.587000 775943 torch/distributed/run.py:852] 
W0527 16:17:28.587000 775943 torch/distributed/run.py:852] *****************************************
W0527 16:17:28.587000 775943 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0527 16:17:28.587000 775943 torch/distributed/run.py:852] *****************************************
[W527 16:17:29.674744248 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
  ✓ Inferred config from checkpoint:
      d_in                 = 6
      d_model              = 256
      n_heads              = 4
      ff_dim               = 512

RuntimeError: Training failed with exit code -15

In [22]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Downstream Evaluation & Diagnostics                       ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys

REPO_DIR = "/kaggle/working/Lpatchtst"
os.chdir(REPO_DIR)

cmd = "python3 -u evaluate.py"
print(f"Running command: {cmd}\n")
print("-" * 70)

process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR
)

try:
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n⚠️ Interrupted! Terminating evaluation process…")
    process.terminate()
    process.wait()

rc = process.poll()
if rc != 0:
    raise RuntimeError(f"Evaluation failed with exit code {rc}")
print("\n✅ Evaluation complete successfully!")

Running command: python3 -u evaluate.py

----------------------------------------------------------------------
Loading data from Data/NIFTY 50_30minute.csv…
FeatureEngineer | ewma_span=260 | horizons=[1, 3, 6, 13, 26, 65, 130, 260] | macd_pairs=[(8, 24), (26, 78), (52, 156)]
Feature columns (0): []
Generating Oracle targets…
Target Distribution — Long: 0.268 | Short: 0.215 | Zero: 0.000
Initializing KronosTokenizer (d_in=6)…
  ✓ Loaded weights from model.safetensors (safetensors)

WFV-aligned splits: train_end=31000, val=[31512:34012], test_start=34524, total=36018
ColumnSelectiveScaler — 0 cols:
  NO_SCALE (0): []
  ROBUST   (0): []
[tokenize_full_series] Tokenizing 36018 bars in chunks of 2048…
[tokenize_full_series] Done. coarse=torch.Size([36018]), fine=torch.Size([36018])
Token vocab usage — coarse: 421/1024, fine: 265/1024
Top-5 coarse tokens: torch.return_types.topk(
values=tensor([1438, 1409, 1332, 1013,  961]),
indices=tensor([ 973,  975, 1007,  991,  291]))
Top-5 fine   toke